# EXAMEN IIB   
## Aubertin Ochoa  
## 15/07/2026

Instalacion de dependencias

In [35]:
!pip install pandas numpy sentence-transformers chromadb google-generativeai streamlit tqdm

INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 37.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 49.9 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.6/797.6 kB 35.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 59.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 64.7 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB

In [23]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download("punkt")
nltk.download("stopwords")



[nltk_data] Downloading package punkt to /home/codespace/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [24]:
# Carga del dataset
df1 = pd.read_csv("data/arxiv_data.csv")
df2 = pd.read_csv("data/arxiv_data_210930-054931.csv")

Exploración del dataset

In [25]:
print("Documento arxiv_data.csv")
print("Filas y columnas:")
print(df1.shape)

print("\nColumnas:")
print(df1.columns)

df1.head()

print("\n\n\nDocumento arxiv_data_210930-054931.csv")
print("Filas y columnas:")
print(df2.shape)

print("\nColumnas:")
print(df2.columns)

df2.head()

Documento arxiv_data.csv
Filas y columnas:
(51774, 3)

Columnas:
Index(['titles', 'summaries', 'terms'], dtype='str')



Documento arxiv_data_210930-054931.csv
Filas y columnas:
(56181, 3)

Columnas:
Index(['terms', 'titles', 'abstracts'], dtype='str')


,terms,titles,abstracts
0,['cs.LG'],Multi-Level Attention Pooling for Graph Neural...,Graph neural networks (GNNs) have been widely ...
1,"['cs.LG', 'cs.AI']",Decision Forests vs. Deep Networks: Conceptual...,Deep networks and decision forests (such as ra...
2,"['cs.LG', 'cs.CR', 'stat.ML']",Power up! Robust Graph Convolutional Network v...,Graph convolutional networks (GCNs) are powerf...
3,"['cs.LG', 'cs.CR']",Releasing Graph Neural Networks with Different...,With the increasing popularity of Graph Neural...
4,['cs.LG'],Recurrence-Aware Long-Term Cognitive Network f...,Machine learning solutions for pattern classif...


Cambiar el nombre de las columnas para que todas sean iguales

In [26]:
df1 = df1.rename(columns={
    "summaries": "abstract"
})

df2 = df2.rename(columns={
    "abstracts": "abstract"
})

In [27]:
df = pd.concat([df1, df2], ignore_index=True)

print("Filas y columnas:")
print(df.shape)

print("\nColumnas:")
print(df.columns)

df.head()

Filas y columnas:
(107955, 3)

Columnas:
Index(['titles', 'abstract', 'terms'], dtype='str')


,titles,abstract,terms
0,Survey on Semantic Stereo Matching / Semantic ...,Stereo matching is one of the widely used tech...,"['cs.CV', 'cs.LG']"
1,FUTURE-AI: Guiding Principles and Consensus Re...,The recent advancements in artificial intellig...,"['cs.CV', 'cs.AI', 'cs.LG']"
2,Enforcing Mutual Consistency of Hard Regions f...,"In this paper, we proposed a novel mutual cons...","['cs.CV', 'cs.AI']"
3,Parameter Decoupling Strategy for Semi-supervi...,Consistency training has proven to be an advan...,['cs.CV']
4,Background-Foreground Segmentation for Interio...,"To ensure safety in automated driving, the cor...","['cs.CV', 'cs.LG']"


Buscar valores nulos y eliminar valores incompletos

In [29]:
stemmer = PorterStemmer()

stop_words = set(stopwords.words("english"))

### Fase 1: Continuación - Limpieza, Filtrado y Construcción del Texto Unificado
En esta celda procedemos a realizar la limpieza de datos eliminando registros duplicados (comunes al unir datasets similares) y manejando valores nulos. Posteriormente, concatenamos el título y el abstract en una sola columna limpia. Dado que los modelos de embeddings semánticos modernos dependen de la sintaxis y el contexto natural de las oraciones, preservaremos el texto original sin aplicar Stemming ni remover stop words.

In [36]:
# 1. Eliminación de Duplicados
# Al concatenar dos datasets de arXiv es muy probable que haya papers repetidos por título
df_cleaned = df.drop_duplicates(subset=['titles']).reset_index(drop=True)
print(# Dimensiones tras eliminar duplicados por título: {df_cleaned.shape[0]} #)

# 2. Manejo de Valores Nulos
# Eliminamos cualquier fila que tenga nulos en campos esenciales
df_cleaned = df_cleaned.dropna(subset=['titles', 'abstract']).reset_index(drop=True)

# 3. Limpieza de Texto Básica (Espacios en blanco y saltos de línea molestos)
def clean_basic_text(text):
    if not isinstance(text, str):
        return ""
    # Remover saltos de línea y tabulaciones por espacios individuales
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df_cleaned['titles'] = df_cleaned['titles'].apply(clean_basic_text)
df_cleaned['abstract'] = df_cleaned['abstract'].apply(clean_basic_text)

# 4. Construcción del Texto Enriquecido para el Embedding
# Añadir etiquetas "Title:" y "Abstract:" ayuda al modelo semántico a entender la jerarquía del fragmento
df_cleaned['text_to_embed'] = "Title: " + df_cleaned['titles'] + "\nAbstract: " + df_cleaned['abstract']

SyntaxError: '(' was never closed (4236438355.py, line 4)